# Chapter 7 — Eclipses and ellipsoidal variability

**Goal.** Read a light curve before interpreting its period or assigning a companion mass. Distinguish
occultations from tidal modulation, fold real Gaia epochs, and measure two sources of incompleteness.
This chapter runs independently and downloads its catalogue and epoch-photometry extracts automatically.

A dip records lost light, not a mass. Two similar dips per orbit can make a period search prefer half the
orbital period. A tidally distorted star can also vary twice per orbit without eclipsing. We start with
controlled models, then inspect two bright real systems. These examples are chosen for teaching, not to
represent the population. Mowlavi et al. (2023) describe the Gaia photometric catalogue and its period choices:
https://arxiv.org/abs/2211.00929.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts, attached to a GitHub release of the tutorial. MANIFEST.json lists, for each file,
# its origin (SQL query or URL) and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/gaia-nss-tutorial/releases/download/data-v0.3/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)

## 7.1 What makes the light change?

Model two circular, uniformly bright disks on an edge-on circular orbit. Let their radii be 0.12 and 0.08
in units of the relative semimajor axis, and surface brightnesses 1 and 0.5. Their projected separation is
$|\sin(2\pi\phi)|$. The lost flux is the overlap area times the surface brightness of the star behind.
At conjunction the smaller disk is fully superposed, giving an analytic check on both eclipse depths.
This model omits limb darkening, reflection, spots, tidal distortion and exposure-time averaging.

For comparison, use a separate harmonic toy $F/\bar F=1-0.02\cos(4\pi\phi)$: two maxima and two minima
per orbit. Its amplitude is stipulated, not fitted or translated into a mass. Physical ellipsoidal models
also depend on inclination, stellar radius, mass ratio, surface-brightness physics and dilution; see
Gomel et al. (2021), https://arxiv.org/abs/2008.11209.

In [ ]:
def overlap(d, r1, r2):
    d = np.asarray(d, dtype=float)
    area = np.zeros_like(d)
    area[d <= abs(r1-r2)] = np.pi*min(r1, r2)**2
    partial = (d > abs(r1-r2)) & (d < r1+r2)
    x = d[partial]                          # compute the lens only where neither disk contains the other
    c1 = np.clip((x*x+r1*r1-r2*r2)/(2*x*r1), -1, 1)
    c2 = np.clip((x*x+r2*r2-r1*r1)/(2*x*r2), -1, 1)
    rad = (-x+r1+r2)*(x+r1-r2)*(x-r1+r2)*(x+r1+r2)
    area[partial] = r1*r1*np.arccos(c1)+r2*r2*np.arccos(c2)-0.5*np.sqrt(np.maximum(rad, 0))
    return area

r1, r2, surface2 = 0.12, 0.08, 0.5
normal_flux = np.pi*(r1*r1+surface2*r2*r2)
def eclipse_flux(phase):
    angle = 2*np.pi*np.asarray(phase)
    blocked_brightness = np.where(np.cos(angle) >= 0, 1.0, surface2)
    return 1-overlap(np.abs(np.sin(angle)), r1, r2)*blocked_brightness/normal_flux
ellipsoid = lambda phase: 1-0.02*np.cos(4*np.pi*np.asarray(phase))
phase = np.linspace(-0.2, 1.2, 1401)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(phase, eclipse_flux(phase), label='uniform disks')
ax[1].plot(phase, ellipsoid(phase), color='C1', label='ellipsoidal harmonic toy')
for a in ax:
    a.set(xlabel='orbital phase [cycles]', ylabel='normalised flux'); a.legend()
plt.tight_layout(); plt.show()
# Mathematical limits: the overlap at either conjunction is the full area of the smaller disk.
expected = 1-np.pi*r2*r2*np.array([1, surface2])/normal_flux
assert np.allclose(eclipse_flux([0, 0.5]), expected, rtol=0, atol=1e-12)
assert np.allclose(ellipsoid(phase), ellipsoid(phase+0.5), rtol=0, atol=1e-12)
assert not np.isclose(eclipse_flux(0), eclipse_flux(0.5))

**Figure 7.1.** Normalised flux versus orbital phase for two independent simulations: uniform-disk eclipses
(blue, left) and a prescribed 2% ellipsoidal harmonic (orange, right). Phase is in cycles; flux is dimensionless.
The two eclipse depths differ because the surface brightnesses differ. The harmonic repeats exactly after half
an orbit. A period-search peak at twice the orbital frequency therefore need not identify the orbital period.

## 7.2 Fold observed Gaia photometry

The extract contains the two brightest `EclipsingBinary` sources by mean G in our shipped NSS catalogue.
Keep only finite positive G fluxes with positive errors, and reject both variability and photometry flags.
Do this explicitly: the service's `VALID_DATA=TRUE` response still contains a variability-rejected G point.
Time is barycentric JD in TCB minus 2455197.5 days. We subtract the first good epoch to define an arbitrary
phase zero; this is not a fitted eclipse epoch. Never mix a TCB epoch with a UTC or TDB ephemeris silently.

Compare folds at the NSS orbital period and half that period. Plot normalised flux with propagated reported
flux errors. Those small formal errors do not account for all calibration systematics or intrinsic variability.

In [ ]:
nss = load('nss_two_body')
epoch = load('eb_epoch_examples')
ids = [5895765142704352640, 541801332594262912]
assert epoch.source_id.dtype.kind in 'iu' and set(epoch.source_id) == set(ids)
assert not epoch.duplicated(['source_id', 'transit_id']).any()
good = (~epoch.variability_flag_g_reject & ~epoch.rejected_by_photometry
        & np.isfinite(epoch.g_transit_time) & np.isfinite(epoch.g_transit_flux)
        & np.isfinite(epoch.g_transit_flux_error) & (epoch.g_transit_flux > 0)
        & (epoch.g_transit_flux_error > 0))
lc = epoch[good].copy()
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for row, sid in enumerate(ids):
    source = nss[(nss.source_id == sid) & (nss.nss_solution_type == 'EclipsingBinary')]
    assert len(source) == 1
    P = float(source.period.iloc[0])
    data = lc[lc.source_id == sid].sort_values('g_transit_time')
    t = data.g_transit_time.to_numpy()
    scale = data.g_transit_flux.median()        # display normalisation, not an out-of-eclipse flux estimate
    flux, error = data.g_transit_flux/scale, data.g_transit_flux_error/scale
    for col, factor in enumerate([1, 0.5]):
        ph = ((t-t.min())/(factor*P)) % 1       # remainder maps all epochs to a single cycle
        ax[row, col].errorbar(ph, flux, error, fmt='.', color=f'C{row}', capsize=2)
        ax[row, col].set(title=f'{sid}\nP_fold = {factor*P:.6f} d',
                         xlabel='phase from first good epoch [cycles]', ylabel='G flux / median G flux')
    print(sid, 'good G transits:', len(data), 'catalogue period [d]:', P)
plt.tight_layout(); plt.show()
# Data-regression check, frozen after extraction; not an independent measurement of completeness.
assert lc.groupby('source_id').size().to_dict() == {ids[0]: 67, ids[1]: 55}

**Figure 7.2.** Actual Gaia G-band epoch fluxes for the two identified bright systems (blue, top; orange,
bottom), with one-reported-standard-error bars. Left: the catalogue period; right: half the catalogue period.
The horizontal axes are phase in cycles from each source's first retained epoch; vertical axes are flux divided
by its median. Folding at half the period overlays the two halves of the orbit. These sparse light curves
illustrate the ambiguity; this plot alone does not independently establish either orbital period.

## 7.3 A photometric catalogue is not an orbital-solution catalogue

Gaia's `vari_eclipsing_binary` has 2,184,477 candidates. The more restricted NSS `EclipsingBinary` sample has
86,918 solutions; these are different products (Mowlavi et al. 2023). We have the latter in the tutorial.
A fit that returns an inclination or eccentricity has stronger model assumptions than a variability label.

In [ ]:
eb = nss[nss.nss_solution_type == 'EclipsingBinary']
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].bar(['variability candidates', 'NSS EclipsingBinary'], [2184477, len(eb)], color=['0.6', 'C0'])
ax[0].set(yscale='log', ylabel='sources / solutions'); ax[0].tick_params(axis='x', labelrotation=12)
ax[1].hist(eb.period, bins=np.geomspace(0.1, 1e3, 65), histtype='step')
ax[1].set(xscale='log', yscale='log', xlabel='NSS orbital period [days]', ylabel='solutions per bin')
plt.tight_layout(); plt.show()
assert len(eb) == 86918                    # exact published NSS count, independent catalogue benchmark
assert eb.source_id.is_unique

**Figure 7.3.** Left: the published variability-candidate count (grey) and the independently counted NSS
eclipsing solutions (blue), on a logarithmic count axis. Right: the period distribution of those NSS solutions,
in logarithmic day bins and with logarithmic counts (displayed range 0.1–1000 d). The count matches the published
NSS total exactly. The count ratio is not a detection efficiency: the two processing products have different
requirements, and our extract does not contain all variability candidates.

## 7.4 Geometry and cadence remove systems

For a detached circular binary with isotropic orbital poles, an eclipse occurs when
$|\cos i|<(R_1+R_2)/a$, so its geometric probability is $(R_1+R_2)/a$ (capped at one).
Even an edge-on binary can have no observations in eclipse. A separate cadence experiment uses the first
real light curve's observation times, randomises the eclipse phase, and asks for at least three samples
inside a box-shaped primary eclipse occupying 2% of a cycle. This is a sampling calculation, not Gaia's
selection pipeline. Transit times are clustered, so independent uniformly random phase samples are only
a comparison, not a description of the scan law.

In [ ]:
from scipy.stats import binom
rng = np.random.default_rng(7)
radius_ratio = np.linspace(0.02, 0.4, 20)
n_poles = 50000
cosi = rng.uniform(-1, 1, n_poles)
measured = np.array([np.mean(np.abs(cosi) < r) for r in radius_ratio])
se = np.sqrt(radius_ratio*(1-radius_ratio)/n_poles)
times = lc[lc.source_id == ids[0]].g_transit_time.to_numpy()
periods = np.geomspace(0.3, 100, 60)
width, n_phase = 0.02, 4000
offset = rng.uniform(0, 1, n_phase)
sampled = []
for P in periods:
    phase_grid = ((times-times.min())[:, None]/P+offset[None, :]) % 1
    in_eclipse = np.minimum(phase_grid, 1-phase_grid) < width/2
    sampled.append(np.mean(in_eclipse.sum(axis=0) >= 3))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.7))
ax[0].plot(radius_ratio, radius_ratio, 'k--', label='analytic isotropic probability')
ax[0].errorbar(radius_ratio, measured, se, fmt='.', label='random poles, ±1 SE')
ax[0].set(xlabel='(R1 + R2) / a', ylabel='geometric eclipse probability'); ax[0].legend(fontsize=8)
ax[1].semilogx(periods, sampled, label='actual Gaia epochs, random eclipse phase')
ax[1].axhline(binom.sf(2, len(times), width), color='k', ls='--', label='independent uniform-phase approximation')
ax[1].set(xlabel='trial period [days]', ylabel='P(at least 3 primary-eclipse samples)'); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
# Four binomial standard errors, plus one discrete pole; tolerance chosen before the experiment.
assert np.all(np.abs(measured-radius_ratio) < 4*se+1/n_poles)

**Figure 7.4.** Left: geometric eclipse probability versus dimensionless summed radius, from isotropic
simulated poles (blue points, one-standard-error bars) and the analytic line (black dashed). Right: probability
of sampling a simulated 2%-wide eclipse at least three times, using the first system's Gaia epochs (blue),
against trial period in days; black dashed assumes independent uniform phases. Only the geometry is validated
against an exact analytic probability. The cadence curve is conditional on this one source's sampling.

The light curve, orbital geometry and sampling must all enter an occurrence-rate analysis. A twice-per-orbit
modulation is a useful candidate signal, but neither its period nor its amplitude alone identifies a dark mass.
Chapter 8 adds velocities and astrometry to this picture.

**Exercises.** Make the surface brightnesses equal in Figure 7.1. Which period ambiguity becomes exact?
Add constant third light and derive the diluted depth. Change the eclipse duty cycle in Figure 7.4; why is
that not equivalent to changing only the geometric eclipse probability?

## 7.5 An observed ellipsoidal candidate and a literature check

Gaia DR3 4042390512917208960 is a bright example in Gomel et al. (2023), Table 3 and Figure 9:
https://arxiv.org/abs/2206.06032. Fold at their 0.89522-day period and epoch 2383.8996 (BJD−2455197.5).
Fit a constant plus three sine/cosine harmonics directly to the G magnitudes. This is a geometric description
of the modulation, not a physical mass fit. Use unweighted least squares here; compare its second-harmonic
amplitude with the published 0.1039 mag, allowing 0.01 mag for the different fitting choices and rounded ephemeris.
The paper's 38 G transits supply a second independent catalogue check.

The star remains a **photometric candidate** in this example. Its large modulation does not by itself establish
a compact secondary. The source paper discusses additional photometry and contamination; we reproduce only
the Gaia harmonic amplitude, not its mass-ratio inference.


In [ ]:
ell = load('ellipsoidal_epoch_example')
assert (ell.source_id == 4042390512917208960).all()
use = (~ell.variability_flag_g_reject & ~ell.rejected_by_photometry
       & np.isfinite(ell.g_transit_time) & np.isfinite(ell.g_transit_mag)
       & (ell.g_transit_flux > 0) & (ell.g_transit_flux_error > 0))
ell = ell[use]
P_ell, t0_ell = 0.89522, 2383.8996
ph = ((ell.g_transit_time.to_numpy()-t0_ell)/P_ell) % 1

def harmonics(phases):
    angle = 2*np.pi*np.asarray(phases)
    return np.column_stack([np.ones(len(angle))] +
                           [f(k*angle) for k in (1, 2, 3) for f in (np.cos, np.sin)])

coeff = np.linalg.lstsq(harmonics(ph), ell.g_transit_mag, rcond=None)[0]
amp2 = np.hypot(coeff[3], coeff[4])            # phase-independent amplitude of the second harmonic
sigma_mag = 2.5/np.log(10)*ell.g_transit_flux_error/ell.g_transit_flux  # first-order flux-to-mag errors
grid = np.linspace(0, 1, 501)
fig, ax = plt.subplots(figsize=(8, 3.7))
ax.errorbar(ph, ell.g_transit_mag, sigma_mag, fmt='.', color='C0', label='Gaia G epochs')
ax.plot(grid, harmonics(grid)@coeff, color='C1', label='three-harmonic fit')
ax.set(xlabel='phase at published ephemeris [cycles]', ylabel='G magnitude',
       title='Gaia DR3 4042390512917208960')
ax.invert_yaxis(); ax.legend(); plt.tight_layout(); plt.show()
assert len(ell) == 38                         # published number of G transits, exact agreement
assert abs(amp2-0.1039) < 0.01                # published second-harmonic amplitude [mag]
print(f'Second-harmonic amplitude: {amp2:.4f} mag; published: 0.1039 mag')


**Figure 7.5.** Real G-band magnitudes of the named ellipsoidal candidate folded at the published
period and epoch (blue points with reported one-standard-error bars); orange is our unweighted three-harmonic
fit. Phase is in cycles and magnitude increases downwards. The two broad minima connect the observed data
to the twice-per-orbit harmonic toy. The fitted second-harmonic amplitude reproduces the published value
within the stated tolerance; this is a light-curve benchmark, not confirmation of a black hole or neutron star.


## References

- Mowlavi et al. (2023), *Gaia Data Release 3. The first Gaia catalogue of eclipsing-binary candidates*, A&A 674, A16: https://arxiv.org/abs/2211.00929 ; https://doi.org/10.1051/0004-6361/202245330
- Gomel, Faigler & Mazeh (2021), *Search for Dormant Black Holes in Ellipsoidal Variables I. Revisiting the Expected Amplitudes of the Photometric Modulation*: https://arxiv.org/abs/2008.11209 ; https://doi.org/10.1093/mnras/staa3305
- Gaia DR3 epoch-photometry data model, mirrored by AIP: https://gaia.aip.de/metadata/gaiadr3/epoch_photometry/
- Gomel et al. (2023), *Gaia Data Release 3: Ellipsoidal variables with possible black hole or neutron star secondaries*, A&A 674, A19, Table 3: https://arxiv.org/abs/2206.06032 ; https://doi.org/10.1051/0004-6361/202243626
